# FunctionTransformer and the Log Transform (Day 30)

Work through this notebook and you will be able to check whether a feature looks normally distributed, tame right-skew with a log transform wrapped in `FunctionTransformer`, route it to one column with `ColumnTransformer`, and measure honestly whether it helped.

Why it matters: many models behave better when features are roughly symmetric. Here the test is real — LogisticRegression moves 0.7207 → 0.7374 on logged features.

You need `train_test_split`, `fit`/`predict`, and `accuracy_score` (Day 29), plus histogram and Q-Q intuition (Days 19–20). New tools here are `scipy.stats.probplot`, `cross_val_score`, and `np.log1p`.

The path ahead: normality checks → raw baseline → the transformer (with a real bug to study) → transformed scoreboard → cross-validated evidence → seeing the change → selective routing (with a second real bug) → a reusable helper and a deliberate bad idea.

# What this notebook covers
- **Transforms:** log, reciprocal, square, square root, Box-Cox, Yeo-Johnson (this notebook implements the log transform; the rest are outlook).
- **Core idea:** some algorithms perform better when features are close to normally distributed, so we reshape skewed features toward symmetry.

## How to tell whether data looks normal

Three complementary checks, from glance to number:

Histogram shape — a symmetric bell suggests normal; a long tail suggests skew.

Q-Q plot — sample quantiles against normal quantiles. Points on the 45° reference line mean "looks normal"; systematic bending away means skew.

`pd.Series.skew()` — near 0 is symmetric; large positive is right-skewed.

# Log transform

Use it on right-skewed, non-negative data — like Fare, with many cheap tickets and a few very expensive ones. It compresses large values far more than small ones, boiling a huge scale down to a compact one.

Two hard rules: never on negatives (`log` is undefined there), and prefer `log1p` over `log` whenever zeros exist (`log1p(0) = 0`; `log(0)` is negative infinity).

# Other transforms (outlook, not implemented here)
- **Reciprocal** (`1/x`): even stronger compression, breaks on zeros.
- **Square / square root**: spread out or compress, milder than log/reciprocal.
- **Box-Cox / Yeo-Johnson** (Day-31 outlook): learnable power transforms; Box-Cox needs strictly positive data, Yeo-Johnson accepts any sign.

## Imports and setup — what each library contributes
| Import | Library | Why it is in this notebook |
|---|---|---|
| `numpy as np` | NumPy | Arrays and math used throughout (`log1p`, `sin`, `mean`) |
| `pandas as pd` | Pandas | `read_csv`, DataFrame/Series selection, `fillna`, `skew` idea |
| `scipy.stats as stats` | SciPy statistics | `probplot` draws every Q-Q plot (cells 9–10, 18–19, 22–23, 25) |
| `matplotlib.pyplot as plt` | Matplotlib | Figures, side-by-side subplots, `show` |
| `seaborn as sns` | Seaborn | Histograms (`distplot` — deprecated, correction below) |
| `train_test_split` | sklearn.model_selection | One 80/20 split (cell 8) |
| `accuracy_score` | sklearn.metrics | Fraction correct (cells 12, 16, 20) |
| `cross_val_score` | sklearn.model_selection | 15-fold honest averaging (cells 17, 21, 25–26) |
| `LogisticRegression`, `DecisionTreeClassifier` | sklearn | The two competing models: linear (scale-sensitive) vs tree (threshold-based) |
| `FunctionTransformer` | sklearn.preprocessing | Wraps `log1p`/any function into a transformer (cells 13, 20, 25) |
| `ColumnTransformer` | sklearn.compose | Routes the transform to `Fare` only (cells 20, 25) |
(Every import is used below.)

In [ ]:
import numpy as np
# numpy: arrays + math (log1p, sin, mean). 

import pandas as pd
# pandas: DataFrames, read_csv, Series ops. Used in cells 7-26.


import scipy .stats as stats
# stats.probplot draws the Q-Q plots;

import matplotlib.pyplot as plt
# pyplot builds figures;

import seaborn as sns
# seaborn draws histograms (distplot is deprecated: see correction after cell 10).

from sklearn.model_selection import train_test_split
# Splitting (cell 8), 

from  sklearn.metrics import accuracy_score
# scoring (cells 12, 16, 20), 

from sklearn.model_selection import  cross_val_score
# 15-fold validation (cells 17, 21, 25-26).



from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
# The two competing models: linear (scale-sensitive) vs tree (threshold splits).
# Compared before/after every transform.


from sklearn.preprocessing import FunctionTransformer
from  sklearn.compose import ColumnTransformer
# The Day-30 stars: wrap any function into a transformer; route transformers to selected columns.


Names bound, nothing computed — no output by design. The table above is the dependency map for everything that follows.

## Load a 3-column Titanic slice, repair Age, split into X and y
Watch three things: the raw Windows path, double vs single brackets, and where the imputation mean secretly comes from.

In [ ]:
df = pd.read_csv(r"E:\MACHINE_LEARNING\DATA_SETS\train.csv")
# Load the Titanic CSV (raw string keeps the Windows backslashes literal).

df = df[['Survived' ,'Age', 'Fare' ]]
# Double brackets keep a 3-column DataFrame; from here df means this slice.

df['Age'] = df['Age'].fillna(df['Age'].mean())
# Fill Age gaps with the column mean. Simplification: mean uses ALL rows incl. future test data (strictly, learn it on train only).

X = df.iloc[: , 1 : 3 ]
y =df.iloc[: , 0 ]
# Positional split: X = Age/Fare matrix (891, 2); y = Survived target (891,).


### Line 1 — load the file
```python
df = pd.read_csv(r"E:\MACHINE_LEARNING\DATA_SETS\train.csv")
```
`read_csv` parses the CSV into a DataFrame (the `r` prefix keeps the Windows backslashes literal). From here, `df` means the loaded table.

### Line 2 — keep three columns
```python
df = df[['Survived' ,'Age', 'Fare' ]]
```
A list inside double brackets selects several columns and returns a DataFrame — one name in single brackets would give a Series. Rebinding drops the other nine columns; selecting by name survives column reordering, positional indexing does not.

### Line 3 — repair Age
```python
df['Age'] = df['Age'].fillna(df['Age'].mean())
```
`df['Age']` takes the column as a Series; `.mean()` averages the present values, skipping gaps automatically; `.fillna(...)` substitutes that mean; assignment writes the repaired Series back.

Estimators cannot fit on NaN, so this repair is mandatory — but note its flaw: the mean sees the whole column including future test rows. Strictly, imputation values belong to training data only (Day 29 does it right); here it is a simplification made before the split.

### Lines 4–5 — separate features and target
```python
X = df.iloc[: , 1 : 3 ]
y =df.iloc[: , 0 ]
```
`iloc` selects by position: columns 1–2 (`Age`, `Fare`) form `X` with shape (891, 2); column 0 (`Survived`) forms `y` with shape (891,).

## One split before any learning — the point of no return
Everything fitted later (imputation values, log, models) must treat the test slice as unseen. `random_state=34` makes this exact split reproducible.

In [ ]:
X_train ,X_test , y_train , y_test = train_test_split(X  , y , test_size = .2 ,     random_state =  34)
# 80/20 split, fixed shuffle (34). From here the test slice is locked: nothing may learn from it.


### One split, four objects
`test_size=.2` holds out 20% (~178 rows); `random_state=34` fixes the shuffle so the split reproduces exactly.

From this line the test slice is locked away: anything that learns — statistics, transformers, models — may only touch train. Breaking that rule is data leakage, the recurring theme of cells 14, 17, and 21.

## Does Age look normal? Histogram + Q-Q plot, side by side
Left panel shows the shape; right panel tests it against normality. Note the warning in the output — it matters.

In [ ]:
plt.figure(figsize = (14 , 6) )
plt.subplot(121)
sns.histplot(X_train['Age'])
# Deprecated histogram call: still ran here, will break on seaborn>=0.14 (see corrected cell after cell 10).

plt.title('AGE pdf ')

plt.subplot(122)
stats.probplot( X_train['Age'] , dist="norm" , plot =  plt )
# Q-Q against normal quantiles, drawn on the right axes. Dots on the diagonal = looks normal.

plt.show()


### The calls
`figure`/`subplot` arrange two panels side by side. `distplot` draws Age's histogram — but it is deprecated and already warns in the output below; it still rendered here and will break on seaborn 0.14+. The corrected cell after cell 10 replaces it.

`probplot(X_train['Age'], dist="norm", plot=plt)` draws the Q-Q plot on the right axes: sample quantiles against normal quantiles. Points on the diagonal would mean "looks normal".

### Reading the figure
Left panel: the shape of Age — peak, spread, skew. Right panel: distance from the diagonal measures departure from normality; systematic bending means skew. Compare the panels against each other.

## Same normality check for Fare — expect a different answer
Fares are bounded at 0 with a long expensive tail (right-skew, known since Day 20). Same mechanics as cell 9; read this one for the contrast.

In [ ]:
plt.figure(figsize = (14 , 6) )
plt.subplot(121)
sns.distplot(X_train['Fare'])
# Same deprecated call as cell 9 (corrected version below covers both columns).

plt.title('FARE pdf ')

plt.subplot(122)
stats.probplot( X_train['Fare'] , dist="norm" , plot =  plt )
# Expect bending away from the diagonal at the top: right-skewed fares.

plt.show()


Same mechanics as the previous cell — histogram left, Q-Q right, same deprecation warning. What is new is the expectation: fares pile at 0 with rare huge values, so the histogram should show right-skew and the Q-Q points should bend away from the diagonal at the top. That bend is the visual case for the log transform.

## Corrected version of cells 9–10 (NOT EXECUTED — run it to verify)
Same two-panel check per column, with `histplot` instead of the deprecated `distplot`, and one loop instead of two near-identical cells.

In [ ]:
for col in ["Age", "Fare"]:
    plt.figure(figsize=(14, 6))
    plt.subplot(121)
    sns.histplot(X_train[col], kde=True)  # modern replacement: histogram + density, no deprecation
    plt.title(col + " distribution")
    plt.subplot(122)
    stats.probplot(X_train[col], dist="norm", plot=plt)  # unchanged: the Q-Q logic was already correct
    plt.show()
# Difference vs original: histplot is supported; the loop removes duplication. Interpret the figure exactly as cells 9-10 describe.


## Two fresh models, zero training yet
Creating estimators learns nothing — it only stores (default) settings. Two different model families make the transform experiment meaningful: if both improve, the transform helped the *data*, not one algorithm.

In [ ]:
clf = LogisticRegression()
# Fresh linear estimator (default settings). Nothing learned yet.

clf2 = DecisionTreeClassifier()
# Fresh tree estimator. Two families => the transform is tested on the data, not on one algorithm.


### Two lines, no learning
```python
clf = LogisticRegression()
clf2 = DecisionTreeClassifier()
```
Each line stores a fresh, unfitted estimator under its name (`clf` = linear model, `clf2` = tree). Re-creating them later (cell 15) is deliberate: a used estimator carries fitted state, and the before/after comparison must start clean.

## Baseline scoreboard — raw data, both models
Fit on train, predict on test, score the fraction correct. These two numbers are the bar every later transform must beat.

In [ ]:
clf.fit(X_train , y_train)
clf2.fit( X_train , y_train)
# Learn from TRAIN only.

y_pred = clf.predict(X_test)
y_pred1 = clf2.predict(X_test)
# Apply to the unseen test slice.



print("Accuracy LR" , accuracy_score (y_test , y_pred))
print("Accuracy DT" , accuracy_score(y_test , y_pred1))
# Baselines: LR 0.7207, DT 0.6816. Every transform below must beat these.


### Fit, predict, score — twice
`fit` learns from train; `predict` applies to the unseen test slice; `accuracy_score` reports the fraction correct.

`Accuracy LR 0.7207`, `Accuracy DT 0.6816` — the raw-data baselines every later transform must beat.

## One line that runs nothing — it only remembers a function
`FunctionTransformer` does not touch data at construction. The learning happens in cells 14+.

In [ ]:
trf = FunctionTransformer(func= np.log1p)
# Wrap log1p WITHOUT applying it: func= stores the function for later. Stateless: fit will learn nothing.


### Wrapping, not running
`func=np.log1p` stores a reference to the function. Compare with `np.log1p(X)`, which would compute immediately: the transformer calls the stored function later, inside `fit`/`transform`, which is what lets it compose into pipelines.

`log1p(x) = log(1+x)` behaves like `log` except at zero — `log1p(0) = 0` while `log(0)` is negative infinity. Fares include 0, so `log1p` is the correct choice. Whether fitting learns anything is the subject of the next cell.

## Apply the transform — the test line is the whole lesson
Train line first (correct), test line second (look closely: it refits).

In [ ]:
X_train_trans = trf.fit_transform(X_train)
# Correct: fit (+ nothing to learn) and transform the TRAIN slice.

X_test_trans = trf.fit_transform(X_test)
# BUG (preserved): refits on TEST. Must be trf.transform(X_test). Harmless ONLY because log1p is stateless; see correction below.


### Right on train, wrong on test
The train line is the correct pattern: fit (and transform) where learning is allowed.

The test line refits the transformer on test data. The rule is absolute — test data is only ever `transform`ed with what train learned.

Why the scores below still stand: `log1p` is stateless, so refitting learns nothing and the numbers equal a correct `transform`. For a stateful transformer (imputer, scaler, encoder) this same line would leak test statistics into the evaluation. The habit is wrong even when the damage here is zero.

## Corrected test line (NOT EXECUTED — run it to verify)
One-token fix with a workflow-sized meaning.

In [ ]:
X_test_trans = trf.transform(X_test)  # reuse train-fitted transformer; never fit on test
# Difference vs original: no refitting, so test statistics cannot leak. Here log1p is stateless, so printed scores would be identical -- for an imputer/scaler they would change. Habit matters more than this instance.


## Fresh models for a fair rematch


In [ ]:
clf = LogisticRegression()
# Fresh linear model for the rematch (no leftover fitted state).

clf2 = DecisionTreeClassifier()
# Fresh tree model. Fair comparison needs clean estimators.


### Two lines, same as cell 11, new reason
Refitting the *same* estimator objects would also work technically (fit overwrites state), but fresh objects guarantee no leftover state pollutes the before/after comparison. Cheap insurance, good habit.

## Scoreboard after the log transform — compare with cell 12
Identical protocol (fit → predict → accuracy), only the inputs changed to log-scale.

In [ ]:
clf.fit(X_train_trans , y_train)
clf2.fit( X_train_trans , y_train)
# Train both on the LOGGED features.

y_pred = clf.predict(X_test_trans)
y_pred1 = clf2.predict(X_test_trans)
# Predict the (refit-)transformed test slice.



print("Accuracy LR" , accuracy_score (y_test , y_pred))
print("Accuracy DT" , accuracy_score(y_test , y_pred1))
# Results: LR 0.7374 (was 0.7207), DT 0.7095 (was 0.6816). Both beat baseline.





Both models improve: LogReg 0.7374 (was 0.7207) and Tree 0.7095 (was 0.6816). Compressing Fare's tail gave the linear model a cleaner boundary, and the tree found better thresholds on the compressed scale. (The test side was refit — harmless here per the previous cell.)

## Stronger evidence: 15-fold cross-validation on transformed data
One split can flatter or punish; averaging 15 rotations is stabler. Watch *where* the transform is fitted, though.

In [ ]:
X_transf = trf.fit_transform(X)
# Whole-dataset transform BEFORE validation: leakage pattern (harmless for stateless log1p; Pipeline cell below shows the right shape).


clf = LogisticRegression()

clf2 = DecisionTreeClassifier()

print("LR",np.mean(cross_val_score(clf , X_transf , y , scoring = 'accuracy' ,cv= 15)))

print("DT" , np.mean(cross_val_score(clf2 , X_transf , y , scoring = 'accuracy' ,cv= 15)))
# 15-fold means: LR 0.6770, DT 0.6621. Lower than holdout = different protocol, not a contradiction.


### Stronger evidence, one flaw
`cv=15` rotates fifteen train/test folds and averages accuracy: 0.6770 (LR), 0.6621 (DT). Lower than the holdout numbers is not a contradiction — means over fifteen small folds differ from one 80/20 split.

The flaw: the transform was fitted on all of `X` before folding, so fold boundaries leak (harmless for stateless `log1p`, wrong as a habit). The Pipeline cell below keeps preprocessing inside each fold — the honest shape of this experiment.

## Recommended shape: preprocessing inside the folds (NOT EXECUTED)
Instead of transforming all of `X` first, hand the *unfitted* steps to `cross_val_score` inside a Pipeline — each fold then fits its own copy on its own training part.

In [ ]:
from sklearn.pipeline import Pipeline
pipe = Pipeline(
    [
      (  "prep", ColumnTransformer
                                 (  [("log", FunctionTransformer(np.log1p), ["Fare"])],   remainder="passthrough")   ),
    ("clf", LogisticRegression()),
])
np.mean(cross_val_score(pipe, X, y, scoring="accuracy", cv=15))
# Difference vs cell 17: no pre-fitted transform exists, so folds cannot leak. For stateless log1p the number barely moves; for real preprocessing this is the only honest setup.


## See the effect on Age: Q-Q before vs after log
Left = raw Age quantiles, right = logged Age quantiles, same reference line. Judge by distance to the diagonal.

In [ ]:
plt.figure(figsize = (14 , 6) )
plt.subplot(121)
stats.probplot( X_train['Age'] , dist="norm" , plot =  plt )
# Raw Age quantiles vs normal: reference panel.

plt.title(' AGE  Before LOG  ')

plt.subplot(122)
stats.probplot( X_train_trans['Age'] , dist="norm" , plot =  plt )
# Logged Age quantiles: closer to the diagonal = closer to normal.

plt.title(' AGE  After LOG  ')

plt.show()


Compare the panels by distance to the diagonal: points curving away on the left that sit closer on the right mean the log helped. Age was never strongly skewed, so expect a modest change — Fare (next cell) is where the transform earns its keep.

## Same before/after view for Fare — this is the money plot
Fare was the skewed one, so its right panel should show the biggest straightening if the log worked.

In [ ]:
plt.figure(figsize = (14 , 6) )
plt.subplot(121)
stats.probplot( X_train['Fare'] , dist="norm" , plot =  plt )
# Raw Fare: expect top points arcing off the diagonal (right skew).

plt.title(' FAre  Before LOG  ')

plt.subplot(122)
stats.probplot( X_train_trans['Fare'] , dist="norm" , plot =  plt )
# Logged Fare: expect points hugging the line (skew compressed).

plt.title(' FARE  After LOG  ')

plt.show()


Same comparison, bigger expected effect: raw Fare's top points should arc far off the diagonal along its expensive tail, while logged Fare hugs the line. If so, the accuracy gains in cell 16 have a visible cause.

## Transform ONLY Fare: per-column routing with ColumnTransformer
Blanket log on Age was harmless but pointless; here each column gets its own treatment. Watch the output column order.

In [ ]:
trf2 = ColumnTransformer([('log' , FunctionTransformer(np.log1p) , ['Fare'])] , remainder= 'passthrough')
# Tuple = (name, transformer, columns). Only Fare is logged; remainder='passthrough' keeps Age raw. Output order: logged Fare col 0, Age col 1 (names lost).


X_train_trans2 = trf2.fit_transform(X_train)

X_test_trans2 = trf2.transform(X_test)
# fit on TRAIN, transform TEST: the correct pattern (contrast cell 14).


clf = LogisticRegression()

clf2 = DecisionTreeClassifier()


clf.fit(X_train_trans2 , y_train)
clf2.fit( X_train_trans2 , y_train)

y_pred = clf.predict(X_test_trans2)
y_pred1 = clf2.predict(X_test_trans2)



print("Accuracy LR" , accuracy_score (y_test , y_pred))
print("Accuracy DT" , accuracy_score(y_test , y_pred1))
# Selective routing, selective effects: LR 0.7151 (Age still raw), DT 0.7039 (up from 0.6816).


### One transformer per column
Each tuple is (step name, transformer object, columns it touches) — here only `Fare` is logged, and `remainder='passthrough'` keeps `Age` instead of dropping it.

Output ordering matters: transformed blocks come first and passthrough last, so column 0 is logged-Fare and column 1 is raw Age, names gone (plain numpy array). The `fit_transform`-on-train / `transform`-on-test pattern here is correct — compare with cell 14.

### Selective effects
LR 0.7151 sits under its baseline (Age stayed raw, so its scale still presses on the linear model); DT 0.7039 still beats 0.6816. Per-column control gives per-column consequences — that is the point of routing.

## Cross-validate the selective transform — find the copy-paste bug
Read the two print lines character by character before believing the numbers.

In [ ]:
X_transf = trf2.fit_transform(X)


clf = LogisticRegression()

clf2 = DecisionTreeClassifier()

print("LR",np.mean(cross_val_score(clf , X_transf , y , scoring = 'accuracy' ,cv= 15)))

print("DT" , np.mean(cross_val_score(clf , X_transf , y , scoring = 'accuracy' ,cv= 15)))
# BUG (preserved): second line scores clf, not clf2 -- identical 0.6679 twice proves it. See corrected cell below.


Both lines print `0.6679472693032015`. Two genuinely different algorithms agreeing to sixteen decimals is impossible by chance — identical numbers are the fingerprint of evaluating one model twice. The second line passes `clf` where it means `clf2`: a copy-paste slip. The corrected cell changes one token; the DT number will move (its exact value is unknown until executed).

## Corrected DT line (NOT EXECUTED — run it to verify)
The whole fix is one identifier.

In [ ]:
print("DT", np.mean(cross_val_score(clf2, X_transf, y, scoring="accuracy", cv=15)))
# Difference vs original: evaluates the TREE (clf2), so the printed number will differ from 0.6679. Its exact value is unknown until executed -- not guessed here.


## Re-run of the Age Q-Q comparison (same code as cell 18)
Confirms the pattern is stable. Mechanics explained at cell 18 — not repeated here.

In [ ]:
plt.figure(figsize = (14 , 6) )
plt.subplot(121)
stats.probplot( X_train['Age'] , dist="norm" , plot =  plt )
# Re-run of cell 18 (confirmation).

plt.title(' AGE  Before LOG  ')

plt.subplot(122)
stats.probplot( X_train_trans['Age'] , dist="norm" , plot =  plt )

plt.title(' AGE  After LOG  ')

plt.show()


Same reading as cell 18 (Age Q-Q before/after). Duplicated run, stable pattern.

## Re-run of the Fare Q-Q comparison (same code as cell 19)
Confirms the pattern is stable. Mechanics explained at cell 19 — not repeated here.

In [ ]:
plt.figure(figsize = (14 , 6) )
plt.subplot(121)
stats.probplot( X_train['Fare'] , dist="norm" , plot =  plt )
# Re-run of cell 19 (confirmation).

plt.title(' Fare  Before LOG  ')

plt.subplot(122)
stats.probplot( X_train_trans['Fare'] , dist="norm" , plot =  plt )

plt.title(' Fare  After LOG  ')

plt.show()


Same reading as cell 19 (Fare Q-Q before/after). Duplicated run, stable pattern.

## Generalize: one helper that tries ANY function on Fare
The transform itself becomes a parameter. `np.sin` at the call site is a deliberate bad idea — read cell 26 for the verdict.

In [ ]:
ml
    print("Accuracy", np.mean(cross_val_score(
        clf, X_trans, y, scoring='accuracy', cv=15
    )))
    # 15-fold mean on the transformed full data (same caveat as cell 17).

    plt.figure(figsize=(14, 4))

    plt.subplot(121)
    stats.probplot(X['Fare'], dist='norm', plot=plt)
    plt.title('Fare Before Transform')

    plt.subplot(122)
    stats.probplot(X_trans[:, 0], dist='norm', plot=plt)
    # Column 0 = transformed Fare (output-ordering rule from cell 20).
    plt.title('Fare After Transform')

    plt.show()


### Functions as arguments
`transform` is a function object supplied by the caller — `np.sin` in the next cell. `FunctionTransformer(transform)` wraps whatever arrives, so one helper demonstrates good ideas and bad ones alike.

The rest reuses established mechanics: same routing as cell 20 (so column 0 is transformed Fare — hence `X_trans[:, 0]`), same full-data-fit caveat as cell 17, same before/after Q-Q reading.


## The verdict on sine: a negative result worth more than a positive one
One call, one accuracy, one figure — and a clear lesson about matching the transform to the data.

In [ ]:
apply_transform(np.sin)
# Deliberate bad idea: periodic sine folds distinct fares together. Result below: 0.6185, far under baseline.


`Accuracy 0.6185` lands far below the ~0.72 baseline. `sin` is periodic: distinct fares fold onto identical values, shredding the monotonic signal Fare carried. Transforms must match the data's shape — log compresses skew, sine wraps values. A clean negative result beats ten vague positives.

## Key takeaways (revision)
- **Concepts:** normality checks (histogram, Q-Q, skew); log-family transforms for right-skew; stateless vs stateful transformers; test-discipline (`transform`, never refit); selective routing + output ordering; CV protocols and their leakage shapes.
- **Functions/methods:** `stats.probplot(x, dist="norm", plot=plt)`, `FunctionTransformer(func)`, `ColumnTransformer([(name, trf, cols)], remainder)`, `fit/transform/fit_transform`, `cross_val_score(est, X, y, scoring, cv)`, `accuracy_score`.
- **Patterns:** fresh estimators per comparison; transform-test reuse; functions-as-arguments (`apply_transform(np.sin)`); negative results as evidence (sine 0.6185).
- **Mistakes fixed above:** `fit_transform` on test (cell 14), `clf`-twice (cell 21), `distplot` (cells 9–10), whole-data fit before CV (cell 17).
## Practice questions (answers in Day_30 `topic_notes.md`)
1. Why `log1p` and not `log` for Fare? 2. What does `func=` store vs what does `np.log1p(X)` do? 3. Why is refitting on test harmless here but forbidden in general? 4. After selective routing, which output column is Fare and why? 5. How do identical CV scores expose the cell-21 bug? 6. Why is 0.6185 for sine informative, not a failure?
